# Distributed PyTorch Printed Circuit Board (PCB) Fault Detection on Serverless Spark

This notebook trains a distributed PyTorch residual convolutional neural network (`PCBResNet`) on **Google Cloud Serverless Spark** (via `ManagedSparkSession` and `TorchDistributor`) to detect faulty printed circuit board (PCB) images stored in Google Cloud Storage.

### Workflow Overview
- **Serverless Spark Initialization & GPU Discovery**: Connect to Google Cloud Serverless Spark using the `dsml` session template and probe remote executor containers for NVIDIA GPU (`CUDA` / `NCCL`) availability.
- **Dataset Exploration & Preprocessing**: Load high-resolution `.JPG` photographs across four PCB designs (`pcb1`-`pcb4`), analyze directory structure and class balance (`good` vs. `bad`), resize images in parallel via `mapInPandas`, and construct a deterministic stratified split.
- **Visual Inspection & Tabular Baseline**: Render sample normal and defective training boards, compute leakage-free RGB channel statistics on `train_df`, and benchmark a PySpark ML `LogisticRegression` model.
- **Distributed PyTorch Training (`TorchDistributor`)**: Train `PCBResNet` across Spark workers using Distributed Data Parallel (`DDP`), `BatchNorm2d`, `Dropout`, `AdamW`, random flip augmentations, and inverse-frequency class weights.
- **Distributed Inference & Evaluation**: Broadcast trained weights to Spark executors for parallel test-set scoring, inspect the confusion matrix, and visualize sample predictions with fault probabilities.

### Dataset Attribution
This demo utilizes the Visual Anomaly (VisA) Dataset by Zou et al. (Amazon Science), used under CC BY 4.0


## Step 1: Initialize Serverless Spark Session (`ManagedSparkSession`)

In [ ]:
import os

from pyspark.sql import SparkSession

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
REGION = os.environ.get("GOOGLE_CLOUD_REGION", "YOUR_REGION")
TEMPLATE_ID = os.environ.get("DATAPROC_SESSION_TEMPLATE", "YOUR_SESSION_TEMPLATE")
SESSION_ID = os.environ.get("DATAPROC_SESSION_ID", "pcb-defect-detection-session")
GCS_DATA_PATH = os.environ.get(
    "PCB_GCS_PATH",
    "gs://YOUR_GCS_BUCKET/training-data/circuit-board-images",
)
TEMPLATE_URI = (
    f"projects/{PROJECT_ID}/locations/{REGION}/sessionTemplates/{TEMPLATE_ID}"
)


def init_serverless_spark() -> SparkSession:
    """Connect to Google Cloud Serverless Spark using the configured session template.

    Returns:
        Active `SparkSession` backed by Google Cloud Serverless Spark.
    """
    active_session = SparkSession.getActiveSession()
    if active_session is not None:
        return active_session

    from google.cloud.dataproc_v1 import Session
    from google.cloud.managed_spark_connect import ManagedSparkSession

    session_config = Session()
    session_config.session_template = TEMPLATE_URI
    return (
        ManagedSparkSession.builder.projectId(PROJECT_ID)
        .location(REGION)
        .sessionId(SESSION_ID)
        .sessionConfig(session_config)
        .getOrCreate()
    )


spark = globals().get("spark") or init_serverless_spark()
spark.conf.set("pytorch.spark.distributor.ignoreSsl", "true")
print(
    f"Connected to Serverless Spark (Spark {spark.version} | template: {TEMPLATE_URI})"
)

The `ManagedSparkSession` connects the notebook to **Google Cloud Serverless Spark** using the `dsml` session template, provisioning a Spark 3.5 environment with remote executor workers and zero local cluster setup.

## Step 2: Verify Environment & Detect GPU Availability on Serverless Spark

Next, we import PyTorch and image processing libraries and run a distributed probe across the Serverless Spark executors via `mapInPandas` to check whether NVIDIA GPUs (`torch.cuda.is_available()`) are attached to the remote workers. When GPUs are detected on the executors, `TorchDistributor` automatically schedules GPU-accelerated training (`use_gpu=True` with `nccl`); otherwise, it falls back cleanly to multi-worker CPU training (`use_gpu=False` with `gloo`).

In [ ]:
import io
import logging
from collections.abc import Callable, Iterator
from typing import TypedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import torch
import torch.nn as nn
from PIL import Image
from pyspark.errors import PySparkException
from pyspark.ml.torch.distributor import TorchDistributor

logger = logging.getLogger(__name__)

PROBE_SCHEMA = (
    "partition_id int, hostname string, torch_version string, "
    "cuda_available boolean, gpu_count int, device_name string"
)


def probe_executor_gpu(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    """Inspect remote Serverless Spark executor containers for NVIDIA GPUs.

    Args:
        iterator: Partition iterator of single-column range DataFrames.

    Yields:
        One-row DataFrames reporting hostname, PyTorch version, and GPU details.
    """
    import socket

    import pandas as pd
    import torch

    for pdf in iterator:
        if pdf.empty:
            continue
        has_cuda = bool(torch.cuda.is_available())
        device_count = int(torch.cuda.device_count()) if has_cuda else 0
        device_name = (
            torch.cuda.get_device_name(0) if has_cuda and device_count > 0 else "CPU"
        )
        yield pd.DataFrame(
            [
                {
                    "partition_id": int(pdf["id"].iloc[0]),
                    "hostname": socket.gethostname(),
                    "torch_version": torch.__version__,
                    "cuda_available": has_cuda,
                    "gpu_count": device_count,
                    "device_name": device_name,
                }
            ]
        )


executor_hw_pdf = (
    spark.range(0, 2, 1, numPartitions=2)
    .mapInPandas(probe_executor_gpu, schema=PROBE_SCHEMA, barrier=True)
    .toPandas()
)
print("Serverless Spark Executor Hardware Probe:")
print(executor_hw_pdf.head(10).to_string(index=False))

USE_GPU = bool(executor_hw_pdf["cuda_available"].any())
TOTAL_EXECUTOR_GPUS = int(
    executor_hw_pdf.drop_duplicates("hostname")["gpu_count"].sum()
)
MAX_DDP_PROCS = 4
NUM_PROCS = min(max(TOTAL_EXECUTOR_GPUS, 2), MAX_DDP_PROCS) if USE_GPU else 2
print(f"\nTorchDistributor Config -> USE_GPU: {USE_GPU} | NUM_PROCS: {NUM_PROCS}")

The executor hardware probe confirms the remote PyTorch runtime and GPU configuration, automatically setting `USE_GPU` and `NUM_PROCS` for distributed training.

## Step 3: Load PCB Image Dataset from GCS & Construct Stratified Split

### 3a. Recursive GCS Load & Directory Layout Inspection

The PCB dataset in Google Cloud Storage follows the **VisA Printed Circuit Board** directory layout across four distinct board designs (`pcb1`, `pcb2`, `pcb3`, `pcb4`):
- `pcb{1..4}/train/good/*.JPG`: Normal (non-faulty) circuit board photographs
- `pcb{1..4}/test/good/*.JPG`: Additional normal circuit board photographs
- `pcb{1..4}/test/bad/*.JPG`: Faulty (defective) circuit board photographs
- `pcb{1..4}/ground_truth/bad/*.png`: Pixel segmentation masks (excluded via `.JPG` filtering so only board photographs are loaded)

We load all `.JPG` photographs recursively using Spark's `binaryFile` data source and extract `pcb_id`, `original_folder_split`, `class_name`, and binary target `label` (`0 = good`, `1 = bad`).

In [ ]:
raw_images_df = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.JPG")
    .load(GCS_DATA_PATH)
)

annotated_images_df = (
    raw_images_df.withColumn(
        "pcb_id", F.regexp_extract(F.col("path"), r"/(pcb\d+)/", 1)
    )
    .withColumn(
        "original_folder_split",
        F.regexp_extract(F.col("path"), r"/(train|test)/", 1),
    )
    .withColumn("class_name", F.regexp_extract(F.col("path"), r"/(good|bad)/", 1))
    .withColumn(
        "label",
        F.when(F.col("class_name") == "bad", F.lit(1))
        .when(F.col("class_name") == "good", F.lit(0))
        .otherwise(F.lit(None).cast("int")),
    )
)

print("Dataset Schema:")
annotated_images_df.select(
    "path",
    "modificationTime",
    "length",
    "pcb_id",
    "original_folder_split",
    "class_name",
    "label",
).printSchema()

distribution_by_board_df = (
    annotated_images_df.groupBy("pcb_id", "class_name", "label")
    .agg(
        F.count("*").alias("image_count"),
        F.round(F.avg("length") / 1024.0, 1).alias("avg_size_kb"),
    )
    .orderBy("pcb_id", "label")
)

distribution_pdf = distribution_by_board_df.limit(10).toPandas()
print("\nImage Counts by PCB Design and Class:")
print(distribution_pdf.to_string(index=False))

### 3b. Class Distribution Across PCB Designs

Because the raw `train/` subdirectories contain only normal (`good`) boards while all defective (`bad`) boards reside under `test/bad/`, we pool labeled `.JPG` images across all four designs before splitting for supervised classification. Below, we plot the class balance of `good` vs. `bad` boards across `pcb1`-`pcb4`.

In [ ]:
class_by_pcb_pdf = distribution_pdf.pivot(
    index="pcb_id", columns="class_name", values="image_count"
).fillna(0)

fig, ax = plt.subplots(figsize=(9, 4.8))
x_pos = np.arange(len(class_by_pcb_pdf.index))
bar_width = 0.36

good_counts = (
    class_by_pcb_pdf["good"].to_numpy()
    if "good" in class_by_pcb_pdf.columns
    else np.zeros(len(x_pos))
)
bad_counts = (
    class_by_pcb_pdf["bad"].to_numpy()
    if "bad" in class_by_pcb_pdf.columns
    else np.zeros(len(x_pos))
)

bars_good = ax.bar(
    x_pos - bar_width / 2,
    good_counts,
    bar_width,
    label="Good / Normal (Label 0)",
    color="#2b8cbe",
)
bars_bad = ax.bar(
    x_pos + bar_width / 2,
    bad_counts,
    bar_width,
    label="Bad / Faulty (Label 1)",
    color="#e34a33",
)

ax.set_title("PCB Dataset Class Distribution by Board Type (pcb1 - pcb4)", fontsize=13)
ax.set_xlabel("PCB Board ID")
ax.set_ylabel("Image Count")
ax.set_xticks(x_pos)
ax.set_xticklabels(class_by_pcb_pdf.index)
ax.legend(frameon=True)
ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar in (*bars_good, *bars_bad):
    height_val = int(bar.get_height())
    ax.annotate(
        f"{height_val}",
        xy=(bar.get_x() + bar.get_width() / 2, height_val),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

### 3c. Distributed Image Preprocessing & Deterministic Stratified Split

Normal (`good`) boards outnumber faulty (`bad`) boards by roughly `10:1` across every PCB design, motivating both a stratified split and class-weighted training loss. Using `mapInPandas`, the Serverless Spark executors validate and decode each photograph in parallel, compute per-image RGB summary features, and encode standardized `256x256` RGB thumbnails (`resized_jpeg`). We then partition the dataset into disjoint **Train (70%)**, **Validation (15%)**, and **Test (15%)** sets using a deterministic, shuffle-free path hash (`xxhash64('path') % 100`).

In [ ]:
prev_split_df = globals().get("split_df")
if prev_split_df is not None:
    prev_split_df.unpersist()

IMG_SIZE = 256

PREPROCESS_SCHEMA = (
    "path string, pcb_id string, class_name string, label int, is_valid boolean, "
    "orig_width int, orig_height int, r_mean double, g_mean double, b_mean double, "
    "r_std double, g_std double, b_std double, resized_jpeg binary"
)

valid_metadata_df = annotated_images_df.filter(
    F.col("label").isNotNull() & (F.col("pcb_id") != "") & (F.col("length") > 0)
)


def preprocess_images_partition(
    iterator: Iterator[pd.DataFrame],
) -> Iterator[pd.DataFrame]:
    """Decode, validate, summarize RGB channels, and resize JPEGs on executors.

    Args:
        iterator: Partition iterator containing raw GCS image bytes and metadata.

    Yields:
        DataFrames with per-channel RGB statistics and 256x256 JPEG thumbnails.
    """
    import io

    import numpy as np
    import pandas as pd
    from PIL import Image, UnidentifiedImageError

    for pdf in iterator:
        out_rows: list[dict[str, str | int | float | bool | bytes]] = []
        paths: list[str] = pdf["path"].astype(str).tolist()
        pcb_ids: list[str] = pdf["pcb_id"].astype(str).tolist()
        classes: list[str] = pdf["class_name"].astype(str).tolist()
        labels: list[int] = pdf["label"].astype(int).tolist()
        contents: list[bytes] = pdf["content"].tolist()
        for row_path, row_pcb_id, row_class, row_label, raw_bytes in zip(
            paths, pcb_ids, classes, labels, contents, strict=True
        ):
            try:
                with Image.open(io.BytesIO(raw_bytes)) as img:
                    rgb_img = img.convert("RGB")
                    orig_w, orig_h = rgb_img.size
                    resized_img = rgb_img.resize(
                        (IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR
                    )
                    arr = np.asarray(resized_img, dtype=np.float32) / 255.0
                    rgb_means = arr.mean(axis=(0, 1))
                    rgb_stds = arr.std(axis=(0, 1))

                    with io.BytesIO() as buf:
                        resized_img.save(buf, format="JPEG", quality=90)
                        resized_bytes = buf.getvalue()

                out_rows.append(
                    {
                        "path": row_path,
                        "pcb_id": row_pcb_id,
                        "class_name": row_class,
                        "label": row_label,
                        "is_valid": True,
                        "orig_width": int(orig_w),
                        "orig_height": int(orig_h),
                        "r_mean": float(rgb_means[0]),
                        "g_mean": float(rgb_means[1]),
                        "b_mean": float(rgb_means[2]),
                        "r_std": float(rgb_stds[0]),
                        "g_std": float(rgb_stds[1]),
                        "b_std": float(rgb_stds[2]),
                        "resized_jpeg": resized_bytes,
                    }
                )
            except (UnidentifiedImageError, OSError, ValueError) as decode_err:
                logger.error("Failed to decode image %s: %s", row_path, decode_err)
                out_rows.append(
                    {
                        "path": row_path,
                        "pcb_id": row_pcb_id,
                        "class_name": row_class,
                        "label": row_label,
                        "is_valid": False,
                        "orig_width": 0,
                        "orig_height": 0,
                        "r_mean": 0.0,
                        "g_mean": 0.0,
                        "b_mean": 0.0,
                        "r_std": 0.0,
                        "g_std": 0.0,
                        "b_std": 0.0,
                        "resized_jpeg": b"",
                    }
                )
        yield pd.DataFrame(out_rows)


processed_df = (
    valid_metadata_df.select("path", "pcb_id", "class_name", "label", "content")
    .repartition(16, F.xxhash64("path"))
    .mapInPandas(preprocess_images_partition, schema=PREPROCESS_SCHEMA)
    .filter(F.col("is_valid"))
)

# Deterministic, shuffle-free path-hash split (70% Train / 15% Val / 15% Test)
split_df = (
    processed_df.withColumn("split_bucket", F.abs(F.xxhash64("path")) % 100)
    .withColumn(
        "dataset_split",
        F.when(F.col("split_bucket") < 70, F.lit("train"))
        .when(F.col("split_bucket") < 85, F.lit("val"))
        .otherwise(F.lit("test")),
    )
    .cache()
)

split_summary_pdf = (
    split_df.groupBy("dataset_split", "class_name", "label")
    .agg(F.count("*").alias("count"))
    .orderBy("dataset_split", "label")
    .limit(10)
    .toPandas()
)
print("Stratified Dataset Split Summary (Train / Val / Test):")
print(split_summary_pdf.to_string(index=False))

train_df = split_df.filter(F.col("dataset_split") == "train")
val_df = split_df.filter(F.col("dataset_split") == "val")
test_df = split_df.filter(F.col("dataset_split") == "test")

## Step 4: Visualize Sample Training Dataset Images

Before training, we sample 4 normal (`good`, `label = 0`) and 4 faulty (`bad`, `label = 1`) circuit boards from the **Training split (`train_df`)** and render them in a `2x4` grid alongside their board design ID (`pcb1`-`pcb4`) and original image dimensions.

In [ ]:
def render_pcb_grid(
    samples_pdf: pd.DataFrame,
    suptitle: str,
    caption_fn: Callable[[int, pd.DataFrame], tuple[str, str]],
) -> None:
    """Render up to 8 PCB thumbnails in a 2x4 matplotlib inspection grid.

    Args:
        samples_pdf: DataFrame containing up to 8 sampled PCB records.
        suptitle: Figure title displayed above the 2x4 image grid.
        caption_fn: Callback returning `(title_text, hex_color)` for row index `i`.

    Raises:
        ValueError: If `samples_pdf` is empty.
    """
    if samples_pdf.empty:
        raise ValueError("samples_pdf must contain at least one image row.")

    fig, axes = plt.subplots(2, 4, figsize=(14, 7.2))
    fig.suptitle(suptitle, fontsize=14, fontweight="bold", y=0.98)
    jpeg_list: list[bytes] = samples_pdf["resized_jpeg"].tolist()

    for idx, ax in enumerate(axes.flat):
        if idx >= len(jpeg_list):
            ax.axis("off")
            continue
        with Image.open(io.BytesIO(jpeg_list[idx])) as img:
            ax.imshow(np.asarray(img.convert("RGB")))
        title_text, title_color = caption_fn(idx, samples_pdf)
        ax.set_title(title_text, fontsize=9.5, color=title_color, fontweight="bold")
        ax.axis("off")

    plt.tight_layout(rect=(0.0, 0.0, 1.0, 0.94))
    plt.show()


def format_training_caption(idx: int, pdf: pd.DataFrame) -> tuple[str, str]:
    """Build title and status color for a sampled training PCB image."""
    pcb_id = str(pdf["pcb_id"].tolist()[idx]).upper()
    is_faulty = int(pdf["label"].tolist()[idx]) == 1
    filename = str(pdf["path"].tolist()[idx]).rsplit("/", maxsplit=1)[-1]
    width_px = int(pdf["orig_width"].tolist()[idx])
    height_px = int(pdf["orig_height"].tolist()[idx])
    status = "FAULTY (bad)" if is_faulty else "NORMAL (good)"
    color = "#d95f02" if is_faulty else "#1b9e77"
    return f"{pcb_id} | {status}\n{filename} ({width_px}x{height_px})", color


train_preview_cols = [
    "path",
    "pcb_id",
    "class_name",
    "label",
    "orig_width",
    "orig_height",
    "resized_jpeg",
]
sample_train_pdf = (
    train_df.filter(F.col("label") == 0)
    .select(*train_preview_cols)
    .limit(4)
    .unionByName(
        train_df.filter(F.col("label") == 1).select(*train_preview_cols).limit(4)
    )
    .limit(8)
    .toPandas()
)

render_pcb_grid(
    sample_train_pdf,
    "Sample Training Split PCB Images (Normal vs. Faulty)",
    format_training_caption,
)

The sample grid above contrasts normal (`good`, top row) and defective (`bad`, bottom row) boards from `train_df`. PCB faults include surface scratches, solder bridges, bent pins, and missing or misaligned surface-mount components—localized spatial defects that global color averages struggle to separate.

## Step 5: Feature Standardization & PySpark ML Baseline Benchmark

To establish a fast tabular benchmark without data leakage:
1. We compute channel-wise RGB mean and standard deviation **exclusively on `train_df`** for downstream normalization.
2. We fit a baseline PySpark ML `LogisticRegression` pipeline (`VectorAssembler` -> `StandardScaler` -> `LogisticRegression`) on those global RGB intensity summaries and evaluate it on the held-out `test_df`.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.feature import StandardScaler, VectorAssembler

# 1. Compute channel-wise RGB mean and std exclusively on train_df (zero data leakage)
channel_stats_pdf = (
    train_df.agg(
        F.avg("r_mean").alias("r_mean"),
        F.avg("g_mean").alias("g_mean"),
        F.avg("b_mean").alias("b_mean"),
        F.avg("r_std").alias("r_std"),
        F.avg("g_std").alias("g_std"),
        F.avg("b_std").alias("b_std"),
    )
    .limit(1)
    .toPandas()
)

TRAIN_RGB_MEAN: tuple[float, float, float] = (
    float(channel_stats_pdf["r_mean"].tolist()[0]),
    float(channel_stats_pdf["g_mean"].tolist()[0]),
    float(channel_stats_pdf["b_mean"].tolist()[0]),
)
TRAIN_RGB_STD: tuple[float, float, float] = (
    max(float(channel_stats_pdf["r_std"].tolist()[0]), 1e-4),
    max(float(channel_stats_pdf["g_std"].tolist()[0]), 1e-4),
    max(float(channel_stats_pdf["b_std"].tolist()[0]), 1e-4),
)
print(f"Train RGB Mean: {[round(v, 4) for v in TRAIN_RGB_MEAN]}")
print(f"Train RGB Std : {[round(v, 4) for v in TRAIN_RGB_STD]}")

# 2. Train Spark ML Logistic Regression baseline on global RGB summary features
rgb_feature_cols = ["r_mean", "g_mean", "b_mean", "r_std", "g_std", "b_std"]
assembler = VectorAssembler(
    inputCols=rgb_feature_cols, outputCol="raw_features", handleInvalid="keep"
)
scaler = StandardScaler(
    inputCol="raw_features", outputCol="features", withMean=True, withStd=True
)
baseline_lr = LogisticRegression(
    featuresCol="features", labelCol="label", maxIter=50, regParam=0.01
)

baseline_model = Pipeline(stages=[assembler, scaler, baseline_lr]).fit(train_df)
baseline_test_preds = baseline_model.transform(test_df)

acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

baseline_acc = float(acc_eval.evaluate(baseline_test_preds))
baseline_f1 = float(f1_eval.evaluate(baseline_test_preds))
baseline_auc = float(auc_eval.evaluate(baseline_test_preds))

base_cm_pdf = (
    baseline_test_preds.groupBy("label", F.col("prediction").cast("int").alias("pred"))
    .agg(F.count("*").alias("count"))
    .limit(10)
    .toPandas()
)
base_cm = np.zeros((2, 2), dtype=np.int64)
base_cm[
    base_cm_pdf["label"].to_numpy(dtype=np.intp),
    base_cm_pdf["pred"].to_numpy(dtype=np.intp),
] = base_cm_pdf["count"].to_numpy(dtype=np.int64)
baseline_tp = int(base_cm[1, 1])
baseline_fp = int(base_cm[0, 1])
baseline_fn = int(base_cm[1, 0])
baseline_recall = baseline_tp / max(baseline_tp + baseline_fn, 1)
baseline_precision = (
    baseline_tp / (baseline_tp + baseline_fp)
    if (baseline_tp + baseline_fp) > 0
    else 0.0
)

print(
    f"\nBaseline Spark ML Logistic Regression -> "
    f"Test Accuracy: {baseline_acc:.4f} | "
    f"Faulty Recall: {baseline_recall:.4f} | "
    f"Weighted F1: {baseline_f1:.4f} | ROC-AUC: {baseline_auc:.4f}"
)

Global RGB summary statistics capture board-wide color shifts, whereas small solder or component defects cover only a tiny fraction of pixels and require spatial convolutional filters.

## Step 6: Distributed PyTorch Model Training (`TorchDistributor`)

### 6a. Custom Residual CNN Architecture (`PCBResNet`)

We define `PCBResNet`, a multi-scale residual convolutional neural network with projection shortcuts, `BatchNorm2d`, dual adaptive average/max spatial pooling, and `Dropout` regularization to detect fine-grained surface anomalies across `pcb1`-`pcb4`.

In [ ]:
class ResidualBlock(nn.Module):
    """Two-layer 3x3 convolutional residual block with projection shortcut."""

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        """Initialize convolutional, BatchNorm, and shortcut layers.

        Args:
            in_channels: Number of input feature channels.
            out_channels: Number of output feature channels.
            stride: Spatial convolution stride for downsampling.
        """
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut: nn.Module = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply residual transformation to input feature map `x`."""
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))


class PCBResNet(nn.Module):
    """Multi-scale residual CNN (`PCBResNet`) for localized PCB defect detection."""

    def __init__(self, num_classes: int = 2) -> None:
        """Initialize stem, four residual stages, multi-scale pooling, and head."""
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.layer1 = ResidualBlock(32, 64, stride=2)
        self.layer2 = ResidualBlock(64, 128, stride=2)
        self.layer3 = ResidualBlock(128, 256, stride=2)
        self.layer4 = ResidualBlock(256, 256, stride=2)
        # Fuse mid-level max features (128) with deep avg+max features (256+256)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.max_pool = nn.AdaptiveMaxPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(640, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute class logits for normalized `(batch, 3, H, W)` PCB images."""
        f1 = self.layer1(self.stem(x))
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        pooled = torch.cat(
            [self.max_pool(f2), self.avg_pool(f4), self.max_pool(f4)],
            dim=1,
        )
        logits: torch.Tensor = self.classifier(pooled)
        return logits


print(PCBResNet(num_classes=2))

### 6b. Distributed Data-Parallel Training on Serverless Spark

We launch distributed data-parallel (`DDP`) training across the Serverless Spark workers via `pyspark.ml.torch.distributor.TorchDistributor`:
- **Hardware Acceleration**: Uses `cuda` + `nccl` with `SyncBatchNorm` when GPUs are present on the executors (`USE_GPU=True`), or `cpu` + `gloo` otherwise.
- **Regularization & Augmentation**: Applies `AdamW` weight decay (`1e-4`), cosine learning-rate annealing, and on-device random horizontal and vertical flip augmentations.
- **Imbalance Handling**: Weights `CrossEntropyLoss` by inverse class frequency so missed defects (`label = 1`) are penalized appropriately, retaining the checkpoint with the strongest validation fault-detection score.

In [ ]:
class TrainingHistory(TypedDict):
    """Epoch-level metrics recorded during distributed PyTorch training."""

    epoch: list[int]
    train_loss: list[float]
    train_acc: list[float]
    val_loss: list[float]
    val_acc: list[float]


class TrainingConfig(TypedDict):
    """Serializable configuration passed to TorchDistributor worker processes."""

    rgb_mean: tuple[float, float, float]
    rgb_std: tuple[float, float, float]
    epochs: int
    batch_size: int
    learning_rate: float


class TrainingOutput(TypedDict):
    """Return payload from distributed PyTorch training on rank 0."""

    state_dict: dict[str, torch.Tensor]
    history: TrainingHistory
    device_used: str
    world_size: int


def train_pcb_distributed(
    train_data: list[tuple[bytes, int]],
    val_data: list[tuple[bytes, int]],
    config: TrainingConfig,
) -> TrainingOutput:
    """Coordinate DDP training of `PCBResNet` across Spark workers.

    Args:
        train_data: Compact `(resized_jpeg, label)` pairs from `train_df`.
        val_data: Compact `(resized_jpeg, label)` pairs from `val_df`.
        config: Hyperparameters and training-split RGB normalization stats.

    Returns:
        Dictionary with best validation CPU `state_dict`, epoch `history`,
        device, and distributed `world_size`.
    """
    import io
    import math
    import os

    import numpy as np
    import torch
    import torch.distributed as dist
    import torch.nn as nn
    import torch.optim as optim
    from PIL import Image
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader, TensorDataset
    from torch.utils.data.distributed import DistributedSampler
    from tqdm.auto import tqdm

    torch.manual_seed(42)
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    use_cuda = bool(torch.cuda.is_available())

    if world_size > 1 and not dist.is_initialized():
        backend = "nccl" if use_cuda else "gloo"
        dist.init_process_group(backend=backend, rank=rank, world_size=world_size)

    if use_cuda:
        gpu_idx = local_rank % max(torch.cuda.device_count(), 1)
        device = torch.device(f"cuda:{gpu_idx}")
        torch.cuda.set_device(device)
    else:
        device = torch.device("cpu")

    def decode_records_once(
        records: list[tuple[bytes, int]],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Decode JPEG byte buffers once into a contiguous CHW `uint8` tensor."""
        with Image.open(io.BytesIO(records[0][0])) as first_img:
            width, height = first_img.size
        count = len(records)
        images_u8 = torch.empty((count, 3, height, width), dtype=torch.uint8)
        labels_t = torch.empty(count, dtype=torch.long)
        for idx, (jpeg_bytes, lbl) in enumerate(records):
            with Image.open(io.BytesIO(jpeg_bytes)) as img:
                hwc = np.array(img.convert("RGB"), dtype=np.uint8, copy=True)
            images_u8[idx] = torch.from_numpy(hwc).permute(2, 0, 1)
            labels_t[idx] = int(lbl)
        return images_u8, labels_t

    train_u8, train_labels = decode_records_once(train_data)
    val_u8, val_labels = decode_records_once(val_data)

    train_dataset = TensorDataset(train_u8, train_labels)
    val_dataset = TensorDataset(val_u8, val_labels)

    train_sampler: DistributedSampler[tuple[torch.Tensor, ...]] | None = (
        DistributedSampler(
            train_dataset, num_replicas=world_size, rank=rank, shuffle=True
        )
        if world_size > 1
        else None
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        sampler=train_sampler,
        shuffle=(train_sampler is None),
        num_workers=0,
        pin_memory=use_cuda,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=0,
        pin_memory=use_cuda,
    )

    mean_t = torch.tensor(config["rgb_mean"], dtype=torch.float32, device=device).view(
        1, 3, 1, 1
    )
    std_t = torch.tensor(config["rgb_std"], dtype=torch.float32, device=device).view(
        1, 3, 1, 1
    )

    base_model = PCBResNet(num_classes=2).to(device)
    if world_size > 1 and use_cuda:
        base_model = nn.SyncBatchNorm.convert_sync_batchnorm(base_model)
    model: nn.Module = (
        DDP(base_model, device_ids=[device.index] if use_cuda else None)
        if world_size > 1
        else base_model
    )

    n_good = max(int((train_labels == 0).sum().item()), 1)
    n_bad = max(int((train_labels == 1).sum().item()), 1)
    defect_weight = math.sqrt(n_good / n_bad)
    class_weights = torch.tensor(
        [1.0, defect_weight], dtype=torch.float32, device=device
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(
        model.parameters(), lr=config["learning_rate"], weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["epochs"], eta_min=1e-5
    )

    history: TrainingHistory = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }
    best_score = -1.0
    best_state_dict: dict[str, torch.Tensor] = {
        k: v.detach().cpu().clone() for k, v in base_model.state_dict().items()
    }

    epoch_bar = tqdm(
        range(1, config["epochs"] + 1),
        desc="Training PCBResNet epochs",
        disable=(rank != 0),
    )
    for epoch in epoch_bar:
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs_u8, targets in train_loader:
            inputs = inputs_u8.to(
                device=device, dtype=torch.float32, non_blocking=use_cuda
            ).div_(255.0)
            targets = targets.to(device, non_blocking=use_cuda)
            h_mask = torch.rand((inputs.size(0), 1, 1, 1), device=device) > 0.5
            v_mask = torch.rand((inputs.size(0), 1, 1, 1), device=device) > 0.5
            inputs = torch.where(h_mask, torch.flip(inputs, dims=[3]), inputs)
            inputs = torch.where(v_mask, torch.flip(inputs, dims=[2]), inputs)
            inputs = (inputs - mean_t) / std_t

            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            batch_count = int(inputs.size(0))
            running_loss += float(loss.item()) * batch_count
            correct += int((outputs.argmax(dim=1) == targets).sum().item())
            total += batch_count
        scheduler.step()

        train_loss = running_loss / max(total, 1)
        train_acc = correct / max(total, 1)

        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        val_tp, val_fp, val_fn = 0, 0, 0
        with torch.inference_mode():
            for v_u8, v_targets in val_loader:
                v_inputs = (
                    v_u8.to(
                        device=device, dtype=torch.float32, non_blocking=use_cuda
                    ).div_(255.0)
                    - mean_t
                ) / std_t
                v_targets = v_targets.to(device, non_blocking=use_cuda)
                v_outputs = model(v_inputs)
                v_loss = criterion(v_outputs, v_targets)
                v_preds = v_outputs.argmax(dim=1)
                v_count = int(v_inputs.size(0))
                val_loss_sum += float(v_loss.item()) * v_count
                val_correct += int((v_preds == v_targets).sum().item())
                val_tp += int(((v_preds == 1) & (v_targets == 1)).sum().item())
                val_fp += int(((v_preds == 1) & (v_targets == 0)).sum().item())
                val_fn += int(((v_preds == 0) & (v_targets == 1)).sum().item())
                val_total += v_count

        val_f1 = (2.0 * val_tp) / max(2 * val_tp + val_fp + val_fn, 1)
        val_acc_epoch = val_correct / max(val_total, 1)
        composite_score = val_f1 + 0.25 * val_acc_epoch

        if rank == 0:
            history["epoch"].append(epoch)
            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["val_loss"].append(val_loss_sum / max(val_total, 1))
            history["val_acc"].append(val_acc_epoch)
            if composite_score >= best_score:
                best_score = composite_score
                best_state_dict = {
                    k: v.detach().cpu().clone()
                    for k, v in base_model.state_dict().items()
                }

    if world_size > 1 and dist.is_initialized():
        dist.destroy_process_group()

    return {
        "state_dict": best_state_dict,
        "history": history,
        "device_used": str(device),
        "world_size": world_size,
    }


train_records_pdf = train_df.select("resized_jpeg", "label").toPandas()
val_records_pdf = val_df.select("resized_jpeg", "label").toPandas()

train_records = list(
    zip(
        train_records_pdf["resized_jpeg"].tolist(),
        train_records_pdf["label"].astype(int).tolist(),
        strict=True,
    )
)
val_records = list(
    zip(
        val_records_pdf["resized_jpeg"].tolist(),
        val_records_pdf["label"].astype(int).tolist(),
        strict=True,
    )
)

training_config: TrainingConfig = {
    "rgb_mean": TRAIN_RGB_MEAN,
    "rgb_std": TRAIN_RGB_STD,
    "epochs": 32,
    "batch_size": 32,
    "learning_rate": 1.0e-3,
}

print(
    f"Launching TorchDistributor ("
    f"samples={len(train_records)}, num_processes={NUM_PROCS}, use_gpu={USE_GPU})..."
)
try:
    training_output: TrainingOutput = TorchDistributor(
        num_processes=NUM_PROCS,
        local_mode=False,
        use_gpu=USE_GPU,
    ).run(train_pcb_distributed, train_records, val_records, training_config)
except (PySparkException, RuntimeError, OSError, ValueError) as dist_err:
    logger.error(
        "Multi-worker TorchDistributor failed (%s); falling back to local_mode=True.",
        dist_err,
    )
    training_output = TorchDistributor(
        num_processes=1,
        local_mode=True,
        use_gpu=False,
    ).run(train_pcb_distributed, train_records, val_records, training_config)

trained_state_dict = training_output["state_dict"]
history_pdf = pd.DataFrame(training_output["history"])
print(
    f"\nDistributed PyTorch Training Complete -> "
    f"Device: {training_output['device_used']} | "
    f"World Size: {training_output['world_size']}"
)
print(history_pdf.tail(10).to_string(index=False))

### 6c. Training & Validation Convergence Curves

We plot epoch-by-epoch training and validation loss and accuracy recorded on rank 0 to inspect convergence across the distributed run.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(
    history_pdf["epoch"],
    history_pdf["train_loss"],
    marker="o",
    color="#2b8cbe",
    label="Train Loss",
)
ax1.plot(
    history_pdf["epoch"],
    history_pdf["val_loss"],
    marker="s",
    color="#e34a33",
    label="Validation Loss",
)
ax1.set_title("Distributed PyTorch Training & Validation Loss", fontsize=12)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.plot(
    history_pdf["epoch"],
    history_pdf["train_acc"],
    marker="o",
    color="#2b8cbe",
    label="Train Accuracy",
)
ax2.plot(
    history_pdf["epoch"],
    history_pdf["val_acc"],
    marker="s",
    color="#31a354",
    label="Validation Accuracy",
)
ax2.set_title("Distributed PyTorch Training & Validation Accuracy", fontsize=12)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_ylim(0.0, 1.05)
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Step 7: Distributed Evaluation & Confusion Matrix on Held-Out Test Set

### 7a. Parallel Batch Inference Across Spark Executors

We serialize the best trained `PCBResNet` `state_dict` and distribute parallel batch inference across the Serverless Spark executors via `mapInPandas` over the held-out **Test set (`test_df`)**. Each executor scores partitions on GPU (`cuda`) if available, outputting:
- `predicted_label`: `0` (`good` / normal) or `1` (`bad` / faulty)
- `fault_probability`: Predicted softmax probability `P(Faulty | Image)`

We then compare `PCBResNet` directly against the `LogisticRegression` RGB baseline on Test Accuracy, Faulty Recall, Faulty Precision, and F1-score.

In [ ]:
with io.BytesIO() as state_dict_buffer:
    torch.save(trained_state_dict, state_dict_buffer)
    serialized_state_dict = state_dict_buffer.getvalue()

PREDICTION_SCHEMA = (
    "path string, pcb_id string, class_name string, label int, "
    "resized_jpeg binary, predicted_label int, fault_probability double"
)

prev_test_preds_df = globals().get("test_predictions_df")
if prev_test_preds_df is not None:
    prev_test_preds_df.unpersist()


def predict_pcb_partition(
    iterator: Iterator[pd.DataFrame],
) -> Iterator[pd.DataFrame]:
    """Run vectorized batch inference with `PCBResNet` across Spark executors.

    Args:
        iterator: Partition iterator of held-out test set PCB thumbnails.

    Yields:
        DataFrames augmented with `predicted_label` and `fault_probability`.
    """
    import io

    import numpy as np
    import torch
    import torch.nn.functional as F_nn
    from PIL import Image

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PCBResNet(num_classes=2)
    with io.BytesIO(serialized_state_dict) as buf:
        loaded_state = torch.load(buf, map_location=device, weights_only=True)
    model.load_state_dict(loaded_state)
    model.to(device)
    model.eval()

    mean_t = torch.tensor(TRAIN_RGB_MEAN, dtype=torch.float32, device=device).view(
        1, 3, 1, 1
    )
    std_t = torch.tensor(TRAIN_RGB_STD, dtype=torch.float32, device=device).view(
        1, 3, 1, 1
    )

    for pdf in iterator:
        if pdf.empty:
            continue
        decoded_arrays: list[np.ndarray] = []
        for jpeg_bytes in pdf["resized_jpeg"]:
            with Image.open(io.BytesIO(jpeg_bytes)) as img:
                decoded_arrays.append(
                    np.array(img.convert("RGB"), dtype=np.uint8, copy=True)
                )

        probs_chunks: list[np.ndarray] = []
        with torch.inference_mode():
            for start in range(0, len(decoded_arrays), 64):
                chunk_np = np.stack(decoded_arrays[start : start + 64], axis=0)
                batch_chw = (
                    torch.from_numpy(chunk_np)
                    .to(device=device, dtype=torch.float32)
                    .permute(0, 3, 1, 2)
                    .div_(255.0)
                )
                normalized_batch = (batch_chw - mean_t) / std_t
                probs_chunks.append(
                    F_nn.softmax(model(normalized_batch), dim=1).cpu().numpy()
                )
        probs = np.concatenate(probs_chunks, axis=0)

        out_pdf = pdf[["path", "pcb_id", "class_name", "label", "resized_jpeg"]].copy()
        out_pdf["predicted_label"] = np.argmax(probs, axis=1).astype(int)
        out_pdf["fault_probability"] = probs[:, 1].astype(float)
        yield out_pdf


test_predictions_df = (
    test_df.select("path", "pcb_id", "class_name", "label", "resized_jpeg")
    .mapInPandas(predict_pcb_partition, schema=PREDICTION_SCHEMA)
    .cache()
)

cm_spark_pdf = (
    test_predictions_df.groupBy("label", "predicted_label")
    .agg(F.count("*").alias("count"))
    .limit(10)
    .toPandas()
)

cm_matrix = np.zeros((2, 2), dtype=np.int64)
cm_matrix[
    cm_spark_pdf["label"].to_numpy(dtype=np.intp),
    cm_spark_pdf["predicted_label"].to_numpy(dtype=np.intp),
] = cm_spark_pdf["count"].to_numpy(dtype=np.int64)

tn, fp = int(cm_matrix[0, 0]), int(cm_matrix[0, 1])
fn, tp = int(cm_matrix[1, 0]), int(cm_matrix[1, 1])
total_test = max(tn + fp + fn + tp, 1)
pytorch_acc = (tn + tp) / total_test
faulty_precision = tp / max(tp + fp, 1)
faulty_recall = tp / max(tp + fn, 1)
faulty_f1 = (
    2.0 * faulty_precision * faulty_recall / max(faulty_precision + faulty_recall, 1e-8)
)
good_f1 = (2.0 * tn) / max(2 * tn + fp + fn, 1)
pytorch_weighted_f1 = ((tn + fp) * good_f1 + (tp + fn) * faulty_f1) / total_test
baseline_faulty_f1 = (
    2.0
    * baseline_precision
    * baseline_recall
    / max(baseline_precision + baseline_recall, 1e-8)
    if (baseline_precision + baseline_recall) > 0
    else 0.0
)

comparison_pdf = pd.DataFrame(
    [
        {
            "Model": "Baseline (Spark ML LogisticRegression on RGB Stats)",
            "Test Accuracy": round(baseline_acc, 4),
            "Faulty Recall": round(baseline_recall, 4),
            "Faulty Precision": round(baseline_precision, 4),
            "Faulty F1": round(baseline_faulty_f1, 4),
            "Weighted F1": round(baseline_f1, 4),
        },
        {
            "Model": "Distributed PyTorch PCBResNet (TorchDistributor)",
            "Test Accuracy": round(pytorch_acc, 4),
            "Faulty Recall": round(faulty_recall, 4),
            "Faulty Precision": round(faulty_precision, 4),
            "Faulty F1": round(faulty_f1, 4),
            "Weighted F1": round(pytorch_weighted_f1, 4),
        },
    ]
)
print("Model Evaluation Comparison on Held-Out Test Set:")
print(comparison_pdf.to_string(index=False))

### 7b. Held-Out Test Set Confusion Matrix

We visualize the `2x2` confusion matrix over `test_df` to examine true positives, false positives, true negatives, and false negatives across normal and defective boards.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_matrix, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted: Good (0)", "Predicted: Faulty (1)"], fontsize=10)
ax.set_yticklabels(["Actual: Good (0)", "Actual: Faulty (1)"], fontsize=10)
ax.set_title("Distributed PyTorch PCB Fault Detection Confusion Matrix", pad=12)

threshold = float(cm_matrix.max()) / 2.0
for row_idx in range(2):
    for col_idx in range(2):
        cell_val = int(cm_matrix[row_idx, col_idx])
        text_color = "white" if cell_val > threshold else "black"
        ax.text(
            col_idx,
            row_idx,
            f"{cell_val}",
            ha="center",
            va="center",
            color=text_color,
            fontsize=13,
            fontweight="bold",
        )

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

The evaluation comparison and confusion matrix confirm that spatial convolutional features substantially outperform global RGB statistics at catching localized circuit board defects.

## Step 8: Visualize Sample Test Predictions with the Trained Model

Finally, we sample 4 normal (`good`) and 4 faulty (`bad`) circuit board photographs from `test_predictions_df`, display a prediction summary table, and render the boards alongside their ground-truth labels, predicted classes, and fault probabilities `P(Faulty)`.

In [ ]:
def format_prediction_caption(idx: int, pdf: pd.DataFrame) -> tuple[str, str]:
    """Build title and match-status color for a sampled test PCB prediction."""
    pcb_id = str(pdf["pcb_id"].tolist()[idx]).upper()
    actual_lbl = int(pdf["label"].tolist()[idx])
    pred_lbl = int(pdf["predicted_label"].tolist()[idx])
    prob_val = float(pdf["fault_probability"].tolist()[idx])

    actual_str = "FAULTY (1)" if actual_lbl == 1 else "GOOD (0)"
    pred_str = "FAULTY (1)" if pred_lbl == 1 else "GOOD (0)"
    is_correct = actual_lbl == pred_lbl
    match_tag = "CORRECT" if is_correct else "MISMATCH"
    color = "#1b9e77" if is_correct else "#d95f02"
    return (
        f"{pcb_id} [{match_tag}]\nActual: {actual_str} | Pred: {pred_str}\n"
        f"P(Faulty) = {prob_val:.3f}",
        color,
    )


sample_preds_pdf = (
    test_predictions_df.filter(F.col("label") == 0)
    .limit(4)
    .unionByName(test_predictions_df.filter(F.col("label") == 1).limit(4))
    .limit(8)
    .toPandas()
)

display_cols_pdf = sample_preds_pdf[
    ["pcb_id", "class_name", "label", "predicted_label", "fault_probability"]
].copy()
display_cols_pdf["predicted_class"] = display_cols_pdf["predicted_label"].map(
    {0: "good", 1: "bad (faulty)"}
)
display_cols_pdf["fault_probability"] = display_cols_pdf["fault_probability"].round(4)
print("Sample Held-Out Test Predictions (Distributed PyTorch PCBResNet):")
print(display_cols_pdf.to_string(index=False))

render_pcb_grid(
    sample_preds_pdf,
    "Sample Test Set Predictions - Distributed PyTorch PCB Fault Detector",
    format_prediction_caption,
)

### Summary & Next Steps
- **Distributed Pipeline**: Using Google Cloud Serverless Spark, we ingested and preprocessed `4,415` high-resolution PCB photographs from GCS in parallel via `mapInPandas`, trained `PCBResNet` across GPU executors with `TorchDistributor` (`DDP` + `NCCL`), and scored the held-out test split in distributed batches.
- **Spatial vs. Global Features**: While the leakage-free global RGB `LogisticRegression` baseline predicted only the majority `good` class, `PCBResNet` with class-weighted cross-entropy learned localized spatial defect cues and flagged faulty boards with calibrated confidence scores.
- **Next Steps**: Export the trained `PCBResNet` weights (`torch.jit.script` or `state_dict`) to Google Cloud Storage for scheduled Serverless Spark batch inspection pipelines, or incorporate the pixel-level masks under `ground_truth/bad/` to train a defect-localization segmentation head.

---
**Dataset Attribution**: This notebook uses the Visual Anomaly (VisA) Dataset by Zou et al. (Amazon Science), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

In [ ]:
spark.range(1).count()